# single agent를 이용한 고객 응대 chatobt 구현

## 환경 설정

In [1]:
# 필요한 모듈 / 클래스 / 함수 임포트
import os
from dotenv import load_dotenv
from pydantic import BaseModel
from typing import Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import CSVLoader
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import add_messages, StateGraph, END

C:\Users\playdata2\AppData\Local\Temp\ipykernel_17172\799049213.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [2]:
# .env file -> API Key 가져오기
file_path='.env'
output=load_dotenv(file_path, override=True)
print(output)

True


## RAG 시스템 기반 설정

### 데이터 로드 및 변환

In [3]:
# 정리: 데이터(csv file) -> CSVLoader(로드 및 Document 자료형으로 변환) 

# 파일 경로 설정
file_path='faq_manual.csv'

# CSVLoader 클래스 호출 -> CSV 파일 로드 및 변환 도구 생성
loader = CSVLoader(
    file_path=file_path,
    encoding="utf-8",
    # 본문(page_content)에 포함할 컬럼 지정
    content_columns=["question", "answer"]
)

# CSVLoader 클래스 전용 함수 -> load() 함수 호출 -> Document로 변환(행별로 생성)
docs = loader.load()

# 결과 확인
print(f"로드된 총 문서 수: {len(docs)}개")
print("-" * 60)
print(f"[첫 번째 Document 본문(page_content)]\n{docs[0].page_content}")
print("-" * 60)
print(f"[첫 번째 Document 메타데이터(metadata)]\n{docs[0].metadata}")

로드된 총 문서 수: 10개
------------------------------------------------------------
[첫 번째 Document 본문(page_content)]
question: 코드크래프터스 아카데미에 어떻게 가입하나요?
answer: 저희 웹사이트 우측 상단의 '회원가입' 버튼을 클릭하시고, 이메일 주소와 비밀번호만 입력하면 간단하게 가입이 완료됩니다. 이메일 인증 후 바로 모든 서비스를 이용하실 수 있습니다.
------------------------------------------------------------
[첫 번째 Document 메타데이터(metadata)]
{'source': 'faq_manual.csv', 'row': 0}


### 텍스트 임베딩 모델 생성

In [ ]:
# 모델 이름 설정
"""
# text-embedding-3-small 성능
  1) 최대 8,192 토큰
  2) 임베딩 크기는 1,536차원 
"""
model_name = "text-embedding-3-small"

# OpenAI 임베딩 모델 생성
model = OpenAIEmbeddings(
    model=model_name
)

### 백터 저장소 및 검색 도구 생성 

In [6]:
# 백터 저장소 저장 경로 설정
chroma_path='chroma_db_rag'

# 이미 생성된 백터 저장소 있다면, 없다면 새로 생성
if os.path.exists(chroma_path) and os.listdir(chroma_path):
    vectorstore = Chroma(
        persist_directory=chroma_path,
        embedding_function=model
    )
    print("백터 저장소 불러오기 성공")
else:
    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=model,
        persist_directory=chroma_path        
    )
    print("백터 저장소 새로 생성 성공")

백터 저장소 불러오기 성공


In [7]:
# 백터 저장소 -> as_retriever 함수 -> 검색 도구 생성
retriever = vectorstore.as_retriever(search_kwargs={'k':1})

In [8]:
# 검색 도구 성능 테스트
user_query="환불은 어떻게?"
output = retriever.invoke(user_query)
print(f'검색 결과: \n{output}')

검색 결과: 
[Document(id='b6db5c08-cd82-4fea-8b57-bccfa4770f5b', metadata={'source': 'faq_manual.csv', 'row': 1}, page_content='question: 수강 신청한 강의를 환불하고 싶어요. 환불 정책은 어떻게 되나요?\nanswer: 네, 환불 가능합니다. 결제일로부터 7일 이내, 그리고 강의 수강률이 5% 미만일 경우 전액 환불해 드리고 있습니다. 고객센터를 통해 문의하시면 신속하게 처리해 드리겠습니다.')]


## RAG 체인 생성

### 프롬프트 생성

In [9]:
# 시스템 프롬프트 생성
system_prompt = """
당신은 친절한 온라인 교육 플랫폼 직원입니다.
제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.

[답변 규칙]
1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.
2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.
3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.

[컨텍스트]
{context}
"""

# 사용자 프롬프트 생성
user_prompt="{question}"

# 전체 프롬프트 생성: chain 생성 -> ChatpromtpTemplate.from_messages() 함수
prompt = ChatPromptTemplate.from_messages(
    messages=[
        ("system", system_prompt),
        ("human", user_prompt)
    ]
)

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="\n당신은 친절한 온라인 교육 플랫폼 직원입니다.\n제공된 [컨텍스트]를 바탕으로 고객의 질문에 정중하게 답변해주세요.\n\n[답변 규칙]\n1. [컨텍스트]의 내용을 바탕으로 자연스럽게 답변을 재구성하거나 그대로 활용하세요.\n2. 답변 시 '답변:'이라는 표현을 사용하지 마세요.\n3. [컨텍스트]에 질문에 대한 정보가 없다면, '죄송하지만 문의하신 내용에 대해서는 아는 바가 없습니다.'라고 답변하세요.\n\n[컨텍스트]\n{context}\n"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


### LLM 생성

In [10]:
# LangChain의 ChatOpenAI 함수 호출, llm 생성
llm = ChatOpenAI(model='gpt-5-nano')

### output parser 생성 

In [11]:
# LLM 출력 분석 -> AIMessage.content -> 문자열 출력
output_parser = StrOutputParser()

### chain 구성

In [12]:
# LCEL 문법 적용 -> "|" 사용 -> chain 구성
rag_chain = prompt | llm | output_parser

## 그래프 상태(State) 정의

In [13]:
# pydantic 라이브러리 -> BaseModel 클래스 -> 커스텀 자료형 생성
class AgentState(BaseModel):
    # 상태 메시지 누적 구조 생성
    messages: Annotated[list[BaseMessage], add_messages] = []

## 그래프 노드 함수 정의

In [14]:
# 내용 정리
"""
# 기능: 사용자의 질문을 받아 벡터 저장소에서 답변의 근거를 검색하고,
  RAG 체인을 실행하여 답변을 생성하는 노드
"""

def rag_node(state: AgentState) -> dict:
    # 입력 받은 상태 확인
    print(f'state: \n{state}')

    # 대화 스토리 -> 최신 사용자 질문 추출
    question = state.messages[-1].content

    # 검색기 retriever 실행 -> 사용자 질문과 가장 유사도가 높은 문서 1개 추출 -> 결과: [Document]
    retrieved_docs = retriever.invoke(question)

    # 추출된 문서(Document) -> 텍스트 추출 -> Document.page_content
    context_text = retrieved_docs[0].page_content

    # rag_chain.invoke() 함수 호출 -> 사용자 질문에 대한 답변(text) 생성
    response = rag_chain.invoke(
        {"context": context_text,
         "question": question}
    ) 

    # 노드 함수 실행의 결과(return) -> dict 자료형 -> 상태 업데이트
    return {"messages":[AIMessage(content=response)]}

## 그래프 구성 및 컴파일

In [15]:
# 상태 기반 그래프 생성(node X, edgeX)
graph = StateGraph(AgentState)

# 그래프에 노드 추가
graph.add_node("rag_node", rag_node)

# 시작점 설정
graph.set_entry_point("rag_node")

# edge 추가: 노드 실행 완료 후 워크플로우가 안전하게 종료(END)되도록 엣지 연결
graph.add_edge("rag_node", END) 

In [17]:
# 컴파일: 그래프를 실행 가능한 구조(시스템)로 변환 -> agent 생성
agent = graph.compile()

## single agent 실행

In [ ]:
# single agent 실행을 위한 while 반복문 정의 및 실행

print("안녕하세요! 코드크래프터스 아카데미 지원 챗봇입니다. 무엇이든 물어보세요! (종료하려면 'exit' 입력)")
print("-" * 80)

## while 반복문 작성
while True:
    try:
        ### 사용자로부터 실시간 질문 입력 받기
        user_input = input("질문: ")

        ### 사용자 질문 -> exit -> 종료
        if user_input.lower() == 'exit':
            print("챗봇을 종료합니다. 이용해 주셔서 감사합니다.")
            break

        ### 사용자 질문 -> text -> AgentState 자료형으로 변환 -> 초기 상태 생성
        initial_state = AgentState(messages=[HumanMessage(content=user_input)])
        # initial_state = {"messages": [HumanMessage(content=user_input)]}

        ### agent 실행 -> 최종 상태 생성 -> rag_chain 실행 결과 -> final_state -> dict -> messages: [HumanMessage, AIMessage]
        final_state = agent.invoke(initial_state)

        ### 최종 상태 -> final_state -> 가장 최신 메시지: AIMessage -> text 추출
        ai_response = final_state["messages"][-1].content
        print(f"[Chatbot Agent 응답] \n{ai_response}")
        print("-" * 80)

    except Exception as e:
        print(f"오류가 발생했습니다: {e}")